In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split

In [ ]:
%pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123

In [ ]:
import dagshub
dagshub.init(repo_owner='bhargavivyshnavi04', repo_name='Swiggy-Delivery-Time-Prediction', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=bb0e045e-7604-463a-96a2-db65c872983e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e10235ebdaaf76a28622b75d0a06db8cccb1f1cffe06e990d23ccaeb7a2f913f




Accessing as bhargavivyshnavi04

Initialized MLflow to track repo "bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction"

Repository bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction initialized!

In [ ]:
# mlflow experiment
import mlflow
mlflow.set_experiment("Exp 06 - Stacking Regressor Hyperparameter Tuning")

2026/08/11 05:51:14 INFO mlflow.tracking.fluent: Experiment with name 'Exp 06 - Stacking Regressor Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/dcf9f91a8d2d4e5c9db9fc955b602908', creation_time=1786427474068, effective_trace_archival_retention=None, experiment_id='9', last_update_time=1786427474068, lifecycle_stage='active', name='Exp 06 - Stacking Regressor Hyperparameter Tuning', tags={}, trace_location=None, workspace='default'>

In [ ]:
from sklearn import set_config

set_config(transform_output="pandas")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/datasets/swiggy_cleaned.csv"

df = pd.read_csv(path)
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,sunny,high,...,INDO,19,3,saturday,1,15.0,11.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,stormy,jam,...,BANG,25,3,friday,0,5.0,19.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,sandstorms,low,...,BANG,19,3,saturday,1,15.0,8.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,sunny,medium,...,COIMB,5,4,tuesday,0,10.0,18.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,cloudy,high,...,CHEN,26,3,saturday,1,15.0,13.0,afternoon,6.210138,medium


In [ ]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [ ]:
# check for missing values

df.isna().sum()

,0
age,1854
ratings,1908
weather,525
traffic,510
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,993
festival,228
city_type,1198


In [ ]:
temp_df = df.copy().dropna()

In [ ]:
# split into X and y

X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45496,35.0,4.2,windy,jam,2,drinks,motorcycle,1.0,no,metropolitian,0,10.0,night,16.600272,very_long
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,0,10.0,morning,1.489846,short
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,0,5.0,afternoon,6.232393,medium


In [ ]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
# transform target column

pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [ ]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [ ]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [ ]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)


preprocessor

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_order', 'type_of_vehicle',
                                  'festival', 'city_type', 'is_weekend',
                                  'order_time_of_day']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [ ]:
# build the pipeline

processing_pipeline = Pipeline(steps=[
                                # ("simple_imputer",simple_imputer),
                                ("preprocess",preprocessor)
                                # ("knn_imputer",knn_imputer)
                            ])

processing_pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_order',
                                                   'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [ ]:
# do data preprocessing

X_train_trans = processing_pipeline.fit_transform(X_train)

X_test_trans = processing_pipeline.transform(X_test)

In [ ]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.5 MB/s eta 0:00:00


In [ ]:
import optuna

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import StackingRegressor

In [ ]:
# build the best models

best_rf_params = {'n_estimators': 479,
                  'criterion': 'squared_error',
                  'max_depth': 17,
                  'min_samples_split': 9,
                  'min_samples_leaf': 2,
                  'max_samples': 0.6603673526197067}

best_lgbm_params = {'n_estimators': 154,
                    'max_depth': 27,
                    'learning_rate': 0.22234435854395157,
                    'subsample': 0.7592213724048168,
                    'min_child_weight': 20,
                    'min_split_gain': 0.004604680609280751,
                    'reg_lambda': 97.8100237909747
                    }

best_rf = RandomForestRegressor(**best_rf_params)

best_lgbm = LGBMRegressor(**best_lgbm_params)

In [ ]:
def objective(trial):

    with mlflow.start_run(nested=True):

        # Select meta-model
        meta_model_name = trial.suggest_categorical(
            "model",
            ["LR", "KNN", "DT"]
        )

        if meta_model_name == "LR":

            meta = LinearRegression()

        elif meta_model_name == "KNN":

            n_neighbors_knn = trial.suggest_int(
                "n_neighbors_knn",
                1,
                15
            )

            weights_knn = trial.suggest_categorical(
                "weights_knn",
                ["uniform", "distance"]
            )

            meta = KNeighborsRegressor(
                n_neighbors=n_neighbors_knn,
                weights=weights_knn,
                n_jobs=-1
            )

        elif meta_model_name == "DT":

            max_depth_dt = trial.suggest_int(
                "max_depth_dt",
                1,
                10
            )

            min_samples_split_dt = trial.suggest_int(
                "min_samples_split_dt",
                2,
                10
            )

            min_samples_leaf_dt = trial.suggest_int(
                "min_samples_leaf_dt",
                1,
                10
            )

            meta = DecisionTreeRegressor(
                max_depth=max_depth_dt,
                min_samples_split=min_samples_split_dt,
                min_samples_leaf=min_samples_leaf_dt,
                random_state=42
            )

        # Log selected meta-model
        mlflow.log_param(
            "meta_model_name",
            meta_model_name
        )

        # Stacking regressor
        stacking_reg = StackingRegressor(
            estimators=[
                ("rf", best_rf),
                ("lgbm", best_lgbm)
            ],
            final_estimator=meta,
            n_jobs=-1
        )

        # Transform target automatically
        model = TransformedTargetRegressor(
            regressor=stacking_reg,
            transformer=pt
        )

        # Fit using ORIGINAL target
        model.fit(
            X_train_trans,
            y_train
        )

        # Predictions are automatically inverse-transformed
        y_pred_test = model.predict(
            X_test_trans
        )

        # MAE
        error = mean_absolute_error(
            y_test,
            y_pred_test
        )

        mlflow.log_metric(
            "MAE",
            error
        )

        # R2
        r2 = r2_score(
            y_test,
            y_pred_test
        )

        mlflow.log_metric(
            "R2",
            r2
        )

        return error

In [ ]:
# create optuna study
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
  # optimize the objective function
  study.optimize(objective,n_trials=20,n_jobs=-1,show_progress_bar=True)

  # log the best parameters
  mlflow.log_params(study.best_params)

  # log the best score
  mlflow.log_metric("best_score", study.best_value)

[I 2026-08-11 06:31:31,660] A new study created in memory with name: no-name-139aacd8-2cb8-4c4f-880e-78307b10275e


  0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🏃 View run redolent-cow-423 at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9/runs/ab54cabf70f643d5813383beeb6fdd16
🧪 View experiment at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9
[I 2026-08-11 06:39:21,222] Trial 1 finished with value: 3.015895995418036 and parameters: {'model': 'LR'}. Best is trial 1 with value: 3.015895995418036.
🏃 View run masked-wolf-950 at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9/runs/baee4f3928bd4be4abba83ca44e3b001
🧪 View experiment at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9
[I 2026-08-11 06:39:29,669] Trial 0 finished with value: 3.0160283827380794 and parameters: {'model': 'LR'}. Best is trial 1 with value: 3.015895995418036.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


...

🏃 View run polite-croc-527 at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9/runs/b680fef8cef2479aa99baf7d9b1c8e06
🧪 View experiment at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9
[I 2026-08-11 07:42:14,467] Trial 16 finished with value: 3.016428996788581 and parameters: {'model': 'LR'}. Best is trial 10 with value: 3.0151956068635686.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🏃 View run flawless-eel-688 at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9/runs/2164182a6d0346569ec8e561e26b289f
🧪 View experiment at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9
[I 2026-08-11 07:42:29,221] Trial 17 finished with value: 3.0181718258309242 and parameters: {'model': 'LR'}. Best is trial 10 with value: 3.0151956068635686.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🏃 View run sneaky-gnat-970 at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9/runs/151cd593a27a4a66970bcec9254f815c
🧪 View experiment at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9
[I 2026-08-11 07:50:09,128] Trial 18 finished with value: 3.0156016291245624 and parameters: {'model': 'LR'}. Best is trial 10 with value: 3.0151956068635686.
🏃 View run zealous-shrike-648 at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9/runs/e35334206cc14fc787add51bc52b6f78
🧪 View experiment at: https://dagshub.com/bhargavivyshnavi04/Swiggy-Delivery-Time-Prediction.mlflow/#/experiments/9
[I 2026-08-11 07:50:20,612] Trial 19 finished with value: 3.0871182444278453 and parameters: {'model': 'DT', 'max_depth_dt': 10, 'min_samples_split_dt': 10, 'min_samples_leaf_dt': 10}. Best is trial 10 with value: 3.0151956068635686.
🏃 View run best_model at: https://dagshub.com/